# Module 8 · Vision & Prompt Templates
Pass images to the model. Build reusable, parameterized prompt templates.

---
> **Setup:** Run the first two cells before anything else.
> API key is loaded automatically from the `.env` file in the parent folder.

In [1]:
%pip install -q google-genai python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os, json, math, time, base64, getpass, re, urllib.request
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()  # picks up modules/.env symlink
api_key = os.environ.get('GEMINI_API_KEY')
if not api_key:
    api_key = getpass.getpass('Paste your Gemini API key: ')

client = genai.Client(api_key=api_key)
MODEL = 'gemma-4-31b-it'
DEFAULT_MAX = 2048

def cfg(**kwargs):
    kwargs.setdefault('max_output_tokens', DEFAULT_MAX)
    kwargs.setdefault('temperature', 0.7)
    return types.GenerateContentConfig(**kwargs)

def get_text(response) -> str:
    if response.text:
        return response.text.strip()
    parts = []
    for cand in (response.candidates or []):
        if cand.content:
            for part in cand.content.parts:
                if not getattr(part, 'thought', False) and part.text:
                    parts.append(part.text)
    return ''.join(parts).strip()

def _call_with_retry(fn, *args, max_retries=5, **kwargs):
    for attempt in range(max_retries):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            msg = str(e)
            if '429' in msg or 'RESOURCE_EXHAUSTED' in msg:
                # parse suggested retry delay from error (e.g. 'retryDelay': '30s')
                m = re.search(r'retry[^0-9]*([0-9]+)s', msg, re.I)
                wait = int(m.group(1)) + 5 if m else 35
                print(f'  ⏳ Rate-limited — waiting {wait}s (attempt {attempt+1}/{max_retries})')
                time.sleep(wait)
            else:
                raise
    raise RuntimeError('Max retries exceeded')

_raw_gen    = client.models.generate_content
_raw_stream = client.models.generate_content_stream
_raw_embed  = client.models.embed_content
_raw_count  = client.models.count_tokens
client.models.generate_content        = lambda *a,**kw: _call_with_retry(_raw_gen,    *a,**kw)
client.models.generate_content_stream = lambda *a,**kw: _call_with_retry(_raw_stream, *a,**kw)
client.models.embed_content           = lambda *a,**kw: _call_with_retry(_raw_embed,  *a,**kw)
client.models.count_tokens            = lambda *a,**kw: _call_with_retry(_raw_count,  *a,**kw)

print('\u2705 Setup complete. Model:', MODEL, '| Retry-on-429: enabled')


✅ Setup complete. Model: gemma-4-31b-it | Retry-on-429: enabled


### 🔌 API Key Test

In [3]:
try:
    _r = client.models.generate_content(
        model=MODEL,
        contents="Reply with exactly the words: Hello LLM course",
        config=cfg(temperature=0.0)
    )
    print("✅ API key working!")
    print("Model says:", get_text(_r))
except Exception as e:
    print("❌ API key error:", e)

✅ API key working!
Model says: Hello LLM course


---
## 14. Multimodal Input (Vision)

Modern LLMs are **multimodal** — they can process images alongside text.

Use cases: image description, OCR from screenshots, chart analysis, photo Q&A, comparing two images.

The image is tokenised (broken into patches) and fed into the same transformer as the text.

In [4]:
# Pass an image to the model — download it first then send as bytes
# (Gemma 4 requires bytes; some models support direct URL references)
image_url = "https://www.python.org/static/community_logos/python-logo-master-v3-TM.png"

with urllib.request.urlopen(image_url) as resp:
    img_bytes = resp.read()

r = client.models.generate_content(
    model=MODEL,
    contents=[
        types.Part.from_bytes(data=img_bytes, mime_type="image/png"),
        "What does this image show? Describe it in one sentence."
    ],
    config=cfg(temperature=0.3)
)
print("Image:", image_url)
print("Model:", get_text(r))

Image: https://www.python.org/static/community_logos/python-logo-master-v3-TM.png
Model: The image shows the official logo for the Python programming language, featuring two intertwined blue and yellow stylized snakes next to the word "python" in grey text.


In [5]:
# The img_bytes variable already holds the image from the cell above.
# This cell shows how to encode as base64 (useful for APIs that accept base64 strings,
# or for storing/caching images locally).
img_b64 = base64.b64encode(img_bytes).decode()
print(f"Base64 string length: {len(img_b64)} chars")
print(f"First 60 chars: {img_b64[:60]}...")

# Use the base64 bytes directly with from_bytes
r = client.models.generate_content(
    model=MODEL,
    contents=[
        types.Part.from_bytes(data=base64.b64decode(img_b64), mime_type="image/png"),
        "What colours appear in this logo?"
    ],
    config=cfg(temperature=0.0)
)
print("\nModel:", get_text(r))

Base64 string length: 111420 chars
First 60 chars: iVBORw0KGgoAAAANSUhEUgAAAlkAAADLCAYAAABdyYYmAAAABHNCSVQICAgI...



Model: The colours that appear in this logo are:

*   **Blue**
*   **Yellow**
*   **Grey**
*   **White** (background)


In [6]:
# ✏️  To use a LOCAL image file, uncomment and set your path:
# with open("my_screenshot.png", "rb") as f:
#     img_bytes = f.read()
# r = client.models.generate_content(
#     model=MODEL,
#     contents=[
#         types.Part.from_bytes(data=img_bytes, mime_type="image/png"),
#         "Describe what you see."
#     ],
#     config=cfg()
# )
# print(get_text(r))
print("(Uncomment and set a local image path to try this)")

(Uncomment and set a local image path to try this)


---
## 15. Prompt Templates & Parameterization

Hard-coding prompts works for experiments. Real applications need **reusable, parameterized prompts** where only the data changes.

Benefits: reusability, consistency, testability, maintainability.

In [7]:
# Level 1: Simple f-string template
def summarize(text: str, max_words: int = 50) -> str:
    r = client.models.generate_content(
        model=MODEL,
        contents=f"Summarize the following text in at most {max_words} words:\n\n{text}",
        config=cfg(temperature=0.3)
    )
    return get_text(r)

article = """
Python's popularity has surged over the past decade, becoming the go-to language for 
data science, AI, web development, and automation. Its readable syntax lowers the 
barrier for beginners, while its rich library ecosystem satisfies expert needs. 
Major tech companies like Google, Netflix, and NASA use it extensively.
"""

print("50-word summary:", summarize(article, max_words=50))
print()
print("20-word summary:", summarize(article, max_words=20))

50-word summary: Python is widely used in AI, data science, and automation due to its readable syntax and extensive libraries. Its versatility appeals to both beginners and experts, making it a staple for major companies like Google and NASA.



20-word summary: Python's accessibility and power make it a leading language for AI, data science, and major tech companies.


In [8]:
# Level 2: Template class — bundles system instruction + prompt + config
class PromptTemplate:
    def __init__(self, template: str, system_instruction: str = None,
                 temperature: float = 0.5, max_output_tokens: int = DEFAULT_MAX):
        self.template = template
        self.system_instruction = system_instruction
        self.temperature = temperature
        self.max_output_tokens = max_output_tokens

    def run(self, **kwargs) -> str:
        prompt = self.template.format(**kwargs)
        r = client.models.generate_content(
            model=MODEL, contents=prompt,
            config=cfg(system_instruction=self.system_instruction,
                       temperature=self.temperature,
                       max_output_tokens=self.max_output_tokens)
        )
        return get_text(r)


# Define reusable templates
translate_template = PromptTemplate(
    template="Translate the following text to {language}:\n\n{text}",
    temperature=0.2
)

code_review_template = PromptTemplate(
    template="Review this {language} code and list any issues:\n\n```{language}\n{code}\n```",
    system_instruction="You are a senior software engineer. Be concise and specific.",
    temperature=0.2
)

# Use the templates
print("=== Translation ===")
print(translate_template.run(language="French", text="Hello, how are you today?"))

print("\n=== Code Review ===")
print(code_review_template.run(language="python", code="""
def divide(a, b):
    return a / b

result = divide(10, 0)
print(result)
"""))

=== Translation ===


Depending on who you are talking to, there are a few ways to say this:

**Formal or to a group:**
> Bonjour, comment allez-vous aujourd'hui ?

**Informal (to a friend or peer):**
> Salut, comment vas-tu aujourd'hui ?

**Casual:**
> Salut, ça va aujourd'hui ?

=== Code Review ===


### Critical Issues
1. **ZeroDivisionError**: The code will crash at runtime because it attempts to divide by zero.

### Engineering Improvements
2. **Lack of Input Validation**: The `divide` function does not check if the divisor `b` is zero before performing the operation.
3. **Missing Type Hints**: The function signature does not specify expected types (e.g., `float` or `int`), reducing maintainability and IDE support.
4. **Missing Documentation**: There is no docstring explaining the function's purpose or behavior.
5. **No Error Handling**: The call to `divide` is not wrapped in a `try-except` block to handle potential exceptions gracefully.

### Recommended Refactor
```python
from typing import Union

def divide(a: Union[int, float], b: Union[int, float]) -> float:
    """Performs division and handles division by zero."""
    if b == 0:
        raise ValueError("The divisor 'b' cannot be zero.")
    return a / b

try:
    result = divide(10, 0)
    print(result)
except (ValueError

In [9]:
# Level 3: Chaining templates — output of one feeds into the next
extract_template = PromptTemplate(
    template="Extract the 3 most important facts from this text, one per line:\n\n{text}",
    temperature=0.1
)

tweet_template = PromptTemplate(
    template="Turn these facts into a single engaging tweet (max 280 chars, no hashtags):\n\n{facts}",
    temperature=0.8
)

news_article = """
Researchers at MIT have developed a new battery technology that charges 10x faster 
than current lithium-ion batteries. The battery uses a novel solid-state electrolyte 
that enables faster ion movement. It has been tested for 1,000 charge cycles with 
less than 5% capacity degradation. Commercial availability is expected by 2026.
"""

facts = extract_template.run(text=news_article)
print("Step 1 — Extracted facts:")
print(facts)

tweet = tweet_template.run(facts=facts)
print(f"\nStep 2 — Tweet ({len(tweet)} chars):")
print(tweet)

Step 1 — Extracted facts:
MIT developed a battery that charges 10x faster than current lithium-ion batteries.
It retains over 95% capacity after 1,000 charge cycles.
Commercial availability is expected by 2026.



Step 2 — Tweet (180 chars):
10x faster charging and virtually no degradation? MIT just developed a battery that retains over 95% capacity after 1,000 cycles. Get ready—it's expected to hit the market by 2026.


---
## 16. Key Takeaways

### Complete concepts cheat-sheet

| Concept | One-liner |
|---|---|
| LLM | Next-token predictor trained on massive text data |
| Token | ~3–4 chars; unit of cost, speed, and context |
| Thinking model | Reasons internally before answering (Gemma 4, o1, etc.) — needs higher `max_output_tokens` |
| Temperature | Randomness: 0 = deterministic, 2 = chaotic |
| top_k | Hard limit on candidate token count |
| top_p | Keep tokens until cumulative probability ≥ p |
| max_output_tokens | Hard ceiling on response length (includes thinking tokens) |
| stop_sequences | Custom strings that halt generation |
| System instruction | Persona/rules set before conversation starts |
| Context window | Total tokens the model can see at once |
| Stateless | Each call is independent; send history every time |
| Zero-shot | Ask without examples |
| Few-shot | Include 2–5 examples to teach format |
| Chain-of-thought | "Think step by step" — improves reasoning |
| Structured output | Force valid JSON matching a schema |
| Streaming | Receive tokens as generated, not all at once |
| Embedding | Dense vector representing the meaning of text |
| Cosine similarity | How semantically close two embeddings are (0–1) |
| Hallucination | Model confidently generating false information |
| Grounding | Provide facts in prompt; instruct model to use only those |
| Chunking | Split long docs to fit within context window |
| candidate_count | Get N independent responses in one call |
| Multimodal | Model processes both text and images |
| Prompt template | Reusable prompt with runtime variables |

### Suggested next experiments

1. **Parameter sweep** — run the same creative prompt at 5 temperatures, plot output length vs temperature
2. **Interactive chatbot** — wrap chat in `while True: user_input = input("You: ")`
3. **Semantic search engine** — embed a set of documents, find top-3 matches for any query
4. **Hallucination detector** — two-step: generate an answer, then verify it
5. **Document Q&A** — chunk a PDF, embed the chunks, retrieve relevant ones, answer questions
6. **Streaming UI** — build a simple Gradio app that streams responses to a web browser

### Resources
- [Gemini API docs](https://ai.google.dev/gemini-api/docs)
- [Gemma model card](https://ai.google.dev/gemma/docs)
- [Google AI Studio](https://aistudio.google.com) — interactive playground
- [Prompt engineering guide](https://ai.google.dev/gemini-api/docs/prompting-strategies)
- [Embeddings guide](https://ai.google.dev/gemini-api/docs/embeddings)